# Insurance Claim Fraud Detection
Project 12 of 100 — Classification Portfolio

## Project Goal
Build an end-to-end classification workflow with Kaggle data loading, exploratory analysis, preprocessing, baseline models, LazyPredict benchmarking, FLAML AutoML, and final interpretation.

## Business Problem
Predict whether an insurance claim is fraudulent using claimant demographics, incident details, vehicle information, and claim amounts.

## Dataset Overview
Insurance claim tabular dataset with mixed policy, incident, and claim-level fields and a fraud-style categorical outcome.

## Practical Notes
Potential leakage is common in claim-fraud datasets because post-investigation features may be present. Columns should be reviewed before trusting the benchmark.

## Environment Setup

In [ ]:
import subprocess, sys
packages = [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "xgboost", "lightgbm", "category_encoders"
]
for package in packages:
    try:
        __import__(package.split("[")[0].replace("-", "_"))
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
print("Packages ready.")

## Imports

In [ ]:
import os
import warnings
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
print("Imports loaded.")

## Configuration

In [ ]:
PROJECT = 'Insurance Claim Fraud Detection'
KAGGLE_SLUG = 'rohitsahoo/insurance-claim'
FILE_HINT = 'insurance'
TARGET_HINTS = ['fraud_reported', 'fraud', 'claim_status']
TEST_SIZE = 0.2
RANDOM_SEED = 42
FLAML_SECONDS = 90
DATA_DIR = pathlib.Path('data') / PROJECT.lower().replace(' ', '_')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT)
print(KAGGLE_SLUG)

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if os.environ.get(key):
        kaggle_ok = True
        print(f"Credential found: {key}")
        break
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    kaggle_ok = True
    print(f"kaggle.json found at {kaggle_json}")
if not kaggle_ok:
    print("WARNING: Kaggle credentials not found.")
else:
    print("Kaggle credentials verified.")

## Download Dataset

In [ ]:
import kagglehub
try:
    dataset_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Downloaded with kagglehub: {dataset_path}")
except Exception as exc:
    print(f"kagglehub failed: {exc}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p {DATA_DIR} --unzip")
    dataset_path = DATA_DIR
csv_files = sorted(dataset_path.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
for file in csv_files[:10]:
    print(file.name, round(file.stat().st_size / 1e6, 2), 'MB')
print(f"CSV count: {len(csv_files)}")

## Load Main Table

In [ ]:
if not csv_files:
    raise FileNotFoundError('No CSV files found in dataset directory.')
selected_file = next((f for f in csv_files if FILE_HINT and FILE_HINT.lower() in f.name.lower()), csv_files[0])
print(f"Loading {selected_file.name}")
df = pd.read_csv(selected_file)
print(df.shape)
print(df.head(3))
print(df.dtypes.head(20))

## Inspect Columns

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(t) for t in df.dtypes],
    'missing_pct': df.isna().mean().round(4),
    'nunique': df.nunique(dropna=False).values,
})
summary.sort_values(['missing_pct', 'nunique'], ascending=[False, False]).head(25)

## Target Selection

In [ ]:
target_col = next((c for c in df.columns if c.lower() in [x.lower() for x in TARGET_HINTS]), None)
if target_col is None:
    target_col = next((c for c in df.columns if any(h.lower() in c.lower() for h in TARGET_HINTS)), None)
if target_col is None:
    candidate_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 20]
    if not candidate_cols:
        raise ValueError('No plausible target column found.')
    target_col = candidate_cols[-1]
print(f"Selected target: {target_col}")
print(df[target_col].value_counts(dropna=False).head(20))

## Basic Cleaning

In [ ]:
work_df = df.copy()
work_df = work_df.drop_duplicates().reset_index(drop=True)
work_df = work_df.loc[:, work_df.isna().mean() < 0.6].copy()
work_df = work_df.dropna(subset=[target_col]).reset_index(drop=True)
if work_df[target_col].dtype == 'O':
    work_df[target_col] = work_df[target_col].astype(str).str.strip()
print(work_df.shape)
print(f"Remaining columns: {len(work_df.columns)}")

## Target Diagnostics

In [ ]:
y_raw = work_df[target_col].copy()
print(y_raw.describe(include='all'))
class_counts = y_raw.value_counts(dropna=False)
print(class_counts.head(20))
fig = px.bar(class_counts.reset_index(), x='index', y=target_col, title='Target Distribution')
fig.show()

## Feature / Target Split

In [ ]:
drop_like_id = [c for c in work_df.columns if c != target_col and ('id' == c.lower() or c.lower().endswith('_id'))]
X = work_df.drop(columns=[target_col] + drop_like_id, errors='ignore').copy()
y = work_df[target_col].copy()
if y.dtype == 'O' or str(y.dtype).startswith('category'):
    label_encoder = LabelEncoder()
    y = pd.Series(label_encoder.fit_transform(y.astype(str)), name=target_col)
    target_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print(target_mapping)
else:
    unique_sorted = sorted(pd.Series(y).dropna().unique().tolist())
    if len(unique_sorted) > 20:
        raise ValueError('Target looks continuous rather than categorical.')
print(X.shape, y.shape)

## Quick EDA

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(f"Numeric columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
if num_cols:
    display(X[num_cols].describe().T.head(20))
if cat_cols:
    display(X[cat_cols].nunique().sort_values(ascending=False).head(20))

## Visual Analysis

In [ ]:
if num_cols:
    sample_num = num_cols[:6]
    X[sample_num].hist(bins=30, figsize=(14, 8))
    plt.tight_layout()
    plt.show()
if len(num_cols) >= 2:
    corr = pd.concat([X[num_cols], y.rename('target_encoded')], axis=1).corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr.iloc[: min(12, len(corr)), : min(12, len(corr))], cmap='coolwarm', center=0)
    plt.title('Correlation Snapshot')
    plt.show()

## Train/Test Split

In [ ]:
stratify_y = y if y.nunique() <= 20 and y.value_counts().min() > 1 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=stratify_y
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3).sort_index())

## Preprocessing Pipeline

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ],
    remainder='drop'
)
print('Preprocessor ready')

## Baseline Models

In [ ]:
def evaluate_classifier(name, model, X_train, X_test, y_train, y_test, requires_proba=True):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1_weighted": f1_score(y_test, pred, average="weighted"),
    }
    if requires_proba and hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(X_test)
            if proba.shape[1] == 2:
                result["roc_auc"] = roc_auc_score(y_test, proba[:, 1])
            else:
                result["roc_auc"] = roc_auc_score(y_test, proba, multi_class="ovr", average="weighted")
        except Exception:
            result["roc_auc"] = np.nan
    else:
        result["roc_auc"] = np.nan
    print(f"{name:30s} acc={result['accuracy']:.4f} f1={result['f1_weighted']:.4f} auc={result['roc_auc']:.4f}")
    return result

In [ ]:
results = []
baselines = [
    ('Dummy Most Frequent', Pipeline([('prep', preprocessor), ('model', DummyClassifier(strategy='most_frequent'))])),
    ('Logistic Regression', Pipeline([('prep', preprocessor), ('model', LogisticRegression(max_iter=2000))])),
    ('Random Forest', Pipeline([('prep', preprocessor), ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1, class_weight='balanced'))])),
]
for name, estimator in baselines:
    results.append(evaluate_classifier(name, estimator, X_train, X_test, y_train, y_test))
results_df = pd.DataFrame(results).sort_values(['f1_weighted', 'accuracy'], ascending=False)
results_df

## LazyPredict Benchmark

In [ ]:
lazy_train = X_train.copy()
lazy_test = X_test.copy()
for col in lazy_train.columns:
    if lazy_train[col].dtype == 'O':
        freq_map = lazy_train[col].astype(str).value_counts(normalize=True).to_dict()
        lazy_train[col] = lazy_train[col].astype(str).map(freq_map).fillna(0)
        lazy_test[col] = lazy_test[col].astype(str).map(freq_map).fillna(0)
lazy_train = lazy_train.fillna(lazy_train.median(numeric_only=True)).fillna(0)
lazy_test = lazy_test.fillna(lazy_train.median(numeric_only=True)).fillna(0)
try:
    lazy = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    lazy_models, _ = lazy.fit(lazy_train, lazy_test, y_train, y_test)
    display(lazy_models.head(15))
except Exception as exc:
    print(f"LazyPredict failed: {exc}")

## FLAML AutoML

In [ ]:
automl_train = X_train.copy()
automl_test = X_test.copy()
for col in automl_train.columns:
    if automl_train[col].dtype == 'O':
        automl_train[col] = automl_train[col].astype(str)
        automl_test[col] = automl_test[col].astype(str)
automl = AutoML()
automl.fit(
    X_train=automl_train,
    y_train=y_train,
    task='classification',
    time_budget=FLAML_SECONDS,
    metric='accuracy',
    seed=RANDOM_SEED,
)
flaml_pred = automl.predict(automl_test)
flaml_result = {
    'model': f"FLAML ({automl.best_estimator})",
    'accuracy': accuracy_score(y_test, flaml_pred),
    'f1_weighted': f1_score(y_test, flaml_pred, average='weighted'),
    'roc_auc': np.nan,
}
results_df = pd.concat([results_df, pd.DataFrame([flaml_result])], ignore_index=True).sort_values(['f1_weighted', 'accuracy'], ascending=False)
results_df

## Best Model Review

In [ ]:
best_row = results_df.iloc[0]
print(best_row)
if 'FLAML' in best_row['model']:
    best_model = automl
    best_pred = flaml_pred
else:
    best_name = best_row['model']
    best_model = dict(baselines)[best_name]
    best_model.fit(X_train, y_train)
    best_pred = best_model.predict(X_test)
print(classification_report(y_test, best_pred))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Error Slice Review

In [ ]:
review_df = X_test.copy()
review_df['actual'] = y_test.values
review_df['predicted'] = best_pred
review_df['is_error'] = (review_df['actual'] != review_df['predicted']).astype(int)
print(review_df['is_error'].mean())
if num_cols:
    display(review_df.groupby('is_error')[num_cols[:5]].mean().T)
review_df.head(10)

## Key Insights
1. Class balance and label definition determine how informative accuracy is.
2. The preprocessing pipeline matters as much as model choice on mixed tabular data.
3. FLAML provides a strong automated reference point, but the baseline models remain important sanity checks.

## Limitations
1. Dataset download and schema can shift over time on Kaggle.
2. Automatic target inference should always be reviewed before trusting results.
3. Some domains need threshold tuning or cost-sensitive metrics beyond weighted F1.

## Next Improvements
1. Add threshold optimization for imbalanced use cases.
2. Add SHAP or permutation importance for explainability.
3. Add probability calibration and business-cost evaluation.

## Final Summary
This notebook builds a reusable but domain-aware classification workflow: it verifies Kaggle access, loads the dataset, infers the target column, cleans features, benchmarks baseline classifiers, compares them with LazyPredict, runs FLAML AutoML, and reviews the best model with confusion-matrix and error-slice analysis.